In [0]:
%run ./04_utils

In [0]:
# =========================================================
# NOTEBOOK : 03 - GOLD LAYER
# PURPOSE  : Business Aggregations + Upsert Gold Tables
# =========================================================

# =========================================================
# 1. IMPORTS
# =========================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, count
import logging

# ✅ 04_utils se import karo — khud likhne ki zaroorat nahi
# from utils_04 import upsert_delta
# =========================================================
# 2. LOGGING
# =========================================================

logger = logging.getLogger("ecommerce_pipeline.gold")

# =========================================================
# 3. PATHS
# =========================================================

SILVER_PATH       = "/Volumes/workspace/default/sales_volume/silver/"
GOLD_CITY_PATH    = "/Volumes/workspace/default/sales_volume/gold/city_sales"
GOLD_PRODUCT_PATH = "/Volumes/workspace/default/sales_volume/gold/product_sales"
GOLD_PAYMENT_PATH = "/Volumes/workspace/default/sales_volume/gold/payment_analysis"
GOLD_CUSTOMER_PATH= "/Volumes/workspace/default/sales_volume/gold/top_customers"

# =========================================================
# 4. GOLD FUNCTION
# =========================================================

def run_gold(spark):

    try:

        logger.info("Starting Gold Layer")

        # --------------------------------------------------
        # Silver se Data Read karo
        # --------------------------------------------------

        silver_df = spark.read.format("delta").load(SILVER_PATH)

        # --------------------------------------------------
        # Business Aggregations
        # --------------------------------------------------

        city_sales = silver_df.groupBy("city") \
            .agg(
                sum("final_revenue").alias("total_revenue"),
                count("order_id").alias("total_orders")
            )

        product_sales = silver_df.groupBy("product") \
            .agg(
                sum("final_revenue").alias("total_revenue"),
                count("order_id").alias("total_orders")
            )

        payment_analysis = silver_df.groupBy("payment_mode") \
            .agg(
                count("order_id").alias("orders"),
                sum("final_revenue").alias("total_sales")
            )

        top_customers = silver_df.groupBy("customer_id") \
            .agg(
                sum("final_revenue").alias("total_spent")
            )

        logger.info("Gold Aggregations Done")

        # --------------------------------------------------
        # ✅ utils se import kiya — yahan seedha use karo
        # --------------------------------------------------

        # upsert_delta(spark, city_sales,       GOLD_CITY_PATH,    "target.city = source.city")
        # upsert_delta(spark, product_sales,    GOLD_PRODUCT_PATH, "target.product = source.product")
        # upsert_delta(spark, payment_analysis, GOLD_PAYMENT_PATH, "target.payment_mode = source.payment_mode")
        # upsert_delta(spark, top_customers,    GOLD_CUSTOMER_PATH,"target.customer_id = source.customer_id")

        city_sales.write.format("delta").mode("overwrite").save(GOLD_CITY_PATH)

        product_sales.write.format("delta").mode("overwrite").save(GOLD_PRODUCT_PATH)

        payment_analysis.write.format("delta").mode("overwrite").save(GOLD_PAYMENT_PATH)

        top_customers.write.format("delta").mode("overwrite").save(GOLD_CUSTOMER_PATH)

        logger.info("Gold Tables Written Successfully")

        # --------------------------------------------------
        # Optimize + Vacuum
        # --------------------------------------------------

        spark.sql(f"OPTIMIZE delta.`{SILVER_PATH}` ZORDER BY (city)")
        spark.sql(f"VACUUM delta.`{SILVER_PATH}` RETAIN 168 HOURS")

        logger.info("Optimize + Vacuum Done")

        # --------------------------------------------------
        # Display Results
        # --------------------------------------------------

        print("============== CITY SALES ==============")
        display(city_sales)

        print("============== PRODUCT SALES ==============")
        display(product_sales)

        print("============== PAYMENT ANALYSIS ==============")
        display(payment_analysis)

        print("============== TOP CUSTOMERS ==============")
        display(top_customers)

        logger.info("Gold Layer Completed Successfully")

    except Exception as e:

        logger.error(f"Gold Layer Failed : {str(e)}")
        raise


# =========================================================
# 5. DIRECT RUN
# =========================================================

if __name__ == "__main__":

    spark = SparkSession.builder \
        .appName("Gold-Layer") \
        .config("spark.sql.shuffle.partitions", "8") \
        .config("spark.databricks.delta.optimizeWrite.enabled", "true") \
        .config("spark.databricks.delta.autoCompact.enabled", "true") \
        .getOrCreate()

    run_gold(spark)